# 20.13 — Distributed training

Distributed training is a negotiation between parallel compute and the communication needed to make that compute agree.

Optimization supplies gradients, while systems design supplies bandwidth, memory, and failure constraints. This notebook simulates data/model/pipeline scaling without running any distributed training.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2017)


def percentile(values, q):
    return float(np.percentile(np.asarray(values, dtype=float), q))


def softmax(logits, temperature=1.0):
    scaled = np.asarray(logits, dtype=float) / temperature
    shifted = scaled - np.max(scaled, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)


def cross_entropy(target_probs, pred_probs):
    clipped = np.clip(pred_probs, 1e-9, 1.0)
    return float(-np.sum(target_probs * np.log(clipped)))


def expected_calibration_error(confidence, correct, bins=10):
    confidence = np.asarray(confidence, dtype=float)
    correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(confidence)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (confidence < hi)
        if hi == 1.0:
            mask = (confidence >= lo) & (confidence <= hi)
        if np.any(mask):
            gap = abs(float(np.mean(confidence[mask])) - float(np.mean(correct[mask])))
            ece += float(np.mean(mask)) * gap
    return ece


def make_f17_workload_ladder(seed=2017):
    local_rng = np.random.default_rng(seed)
    return [
        {
            "rung": "D1",
            "name": "tiny hand-check",
            "requests": 60,
            "precision": "int8",
            "scale": 1.8 / 127.0,
            "shape": (8,),
            "load": 1.0,
            "data": np.array([0.73, -0.40, 1.20, -1.70, 0.05, 0.88, -0.91, 1.55]),
        },
        {
            "rung": "D2",
            "name": "clean tensor workload",
            "requests": 400,
            "precision": "int8",
            "scale": 2.5 / 127.0,
            "shape": (64, 32),
            "load": 1.7,
            "data": local_rng.normal(0.0, 0.65, size=(64, 32)),
        },
        {
            "rung": "D3",
            "name": "outliers and bad calibration",
            "requests": 900,
            "precision": "int8",
            "scale": 1.0 / 127.0,
            "shape": (128, 48),
            "load": 2.4,
            "data": np.vstack([
                local_rng.normal(0.0, 0.45, size=(124, 48)),
                local_rng.normal(0.0, 2.2, size=(4, 48)),
            ]),
        },
        {
            "rung": "D4",
            "name": "digits-like classifier weights",
            "requests": 1600,
            "precision": "mixed int8/fp16",
            "scale": 3.0 / 127.0,
            "shape": (10, 64),
            "load": 3.5,
            "data": local_rng.normal(0.0, 0.85, size=(10, 64)),
        },
        {
            "rung": "D5",
            "name": "production-scale load simulation",
            "requests": 5000,
            "precision": "per-channel int8",
            "scale": 4.0 / 127.0,
            "shape": (256, 128),
            "load": 6.0,
            "data": local_rng.normal(0.0, 0.95, size=(256, 128)),
        },
    ]


def preview_ladder(ladder):
    for rung in ladder:
        data = rung.get("data")
        shape = getattr(data, "shape", rung.get("shape"))
        print(f"{rung['rung']} | {rung['name']} | shape={shape} | requests={rung['requests']} | precision={rung['precision']}")
        print("sample:", np.asarray(data).reshape(-1)[:5])


## The concept, built once (D1)

Use $T_{obs}=T_1/N+T_{comm}$ and $\eta=(T_1/T_{obs})/N$. The lesson's $160$ minute job on $8$ workers has $20$ ideal minutes, $25$ observed minutes, $6.4\times$ speedup, and $0.80$ efficiency.

In [ ]:

def distributed_time(T1, workers, comm):
    observed = T1 / workers + comm
    speedup = T1 / observed
    efficiency = speedup / workers
    return observed, speedup, efficiency

observed, speedup, efficiency = distributed_time(160, 8, 5)
payload_mb = 50_000_000 * 4 / 1_000_000
pipeline_utilization = 12 / (12 + 4 - 1)

print("observed", observed)
print("speedup", speedup)
print("efficiency", efficiency)
print("payload MB", payload_mb)
print("pipeline utilization", pipeline_utilization)

assert observed == 25
assert round(speedup, 1) == 6.4
assert round(efficiency, 2) == 0.80
assert payload_mb == 200
assert round(pipeline_utilization, 3) == 0.800


The evaluator turns those formulas into throughput. It models compute division, synchronization payload, network spikes, stragglers, pipeline bubbles, and overlap.

In [ ]:

def make_distributed_ladder(seed=2017):
    local_rng = np.random.default_rng(seed + 13)
    specs = [
        ("D1", "hand 2-worker job", 2, 4000, 20.0, 1.0, 2, 4),
        ("D2", "8-worker clean scaling", 8, 16000, 160.0, 5.0, 4, 12),
        ("D3", "network spike and straggler", 12, 26000, 240.0, 18.0, 6, 10),
        ("D4", "sampled training-step trace", 24, 60000, 480.0, 40.0, 8, 24),
        ("D5", "production cluster simulation", 64, 220000, 1440.0, 160.0, 12, 32),
    ]
    ladder = []
    for rung, name, workers, examples, single_minutes, comm_minutes, stages, microbatches in specs:
        trace = local_rng.lognormal(mean=math.log(comm_minutes + 1.0), sigma=0.25, size=200)
        ladder.append({
            "rung": rung,
            "name": name,
            "workers": workers,
            "examples": examples,
            "single_minutes": single_minutes,
            "comm_minutes": comm_minutes,
            "stages": stages,
            "microbatches": microbatches,
            "requests": examples,
            "precision": "fp32 gradients",
            "load": workers / 8,
            "data": trace,
        })
    return ladder


def evaluate_distributed_workload(rung, batch_multiplier=1.0, overlap_fraction=0.0):
    compute_minutes = rung["single_minutes"] / rung["workers"]
    comm_minutes = rung["comm_minutes"] / math.sqrt(batch_multiplier)
    effective_comm = comm_minutes * (1.0 - overlap_fraction)
    pipeline_util = rung["microbatches"] / (rung["microbatches"] + rung["stages"] - 1)
    straggler_minutes = percentile(rung["data"], 95) * 0.05
    observed_minutes = (compute_minutes + effective_comm + straggler_minutes) / pipeline_util
    speedup = rung["single_minutes"] / observed_minutes
    throughput = rung["examples"] / (observed_minutes * 60.0)
    efficiency = speedup / rung["workers"]
    return {
        "observed_minutes": observed_minutes,
        "speedup": speedup,
        "throughput": throughput,
        "efficiency": efficiency,
        "pipeline_util": pipeline_util,
        "artifact": rung["data"],
    }


## The dataset ladder

D1 is a hand 2-worker job; D2 is clean 8-worker scaling; D3 injects network spikes; D4 uses sampled step traces; D5 is a production cluster scaling simulation.

In [ ]:

ladder = make_distributed_ladder()
preview_ladder(ladder)


## Run the SAME method across D1–D5

In [ ]:

distributed_results = []
for rung in ladder:
    result = evaluate_distributed_workload(rung)
    result["rung"] = rung["rung"]
    result["name"] = rung["name"]
    distributed_results.append(result)

print("rung | workers | observed_min | speedup | efficiency | throughput")
for rung, result in zip(ladder, distributed_results):
    print(f"{result['rung']} | {rung['workers']} | {result['observed_minutes']:.1f} | {result['speedup']:.2f}x | {result['efficiency']:.2f} | {result['throughput']:.1f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result in zip(axes[0], distributed_results):
    ax.plot(result["artifact"], linewidth=1.0)
    ax.set_title(result["rung"] + " comm trace")
    ax.set_ylabel("minutes")

metric_curve = [result["speedup"] for result in distributed_results]
axes[1, 0].plot([rung["workers"] for rung in ladder], metric_curve, marker="o")
axes[1, 0].set_title("speedup curve")
axes[1, 0].set_xlabel("workers")
for ax, result in zip(axes[1, 1:], distributed_results[1:]):
    ax.bar(["throughput", "eff", "util"], [result["throughput"], result["efficiency"], result["pipeline_util"]])
    ax.set_title(result["rung"] + " operational panel")
plt.tight_layout()


## Pitfall on D5: expecting linear speedup

Linear speedup ignores communication and bubbles. Reproduce the overly optimistic number, then fix with larger batches and communication overlap.

In [ ]:

d5 = ladder[-1]
naive_observed = d5["single_minutes"] / d5["workers"]
naive_speedup = d5["single_minutes"] / naive_observed
actual = evaluate_distributed_workload(d5)
fixed = evaluate_distributed_workload(d5, batch_multiplier=4.0, overlap_fraction=0.45)

print("naive speedup", round(naive_speedup, 1))
print("actual speedup", round(actual["speedup"], 1))
print("fixed speedup", round(fixed["speedup"], 1))
assert actual["speedup"] < naive_speedup
assert fixed["speedup"] > actual["speedup"]


## Evaluate it + Practice

- Metric: track distributed speedup and throughput; compare with the no-skill baseline `single-worker throughput`.
- Cheap sanity check: the 8-worker lesson case should observe 25 minutes.
- Ablation: set communication to zero and compare with reality.
- Failure signals: efficiency falls as workers increase.
- Production guardrail: monitor the metric by rung instead of averaging D1 with D5.

Practice 1: Change one D5 load or precision setting and predict the metric before running.

Practice 2: Add one operational constraint, such as memory budget or p95 latency budget, and mark the first rung that violates it.

Practice 3: Write a one-line launch rule that uses the metric and one pitfall guardrail.